In [ ]:
%load_ext autoreload
%autoreload 2

import functools
import math
from math import pi
import os
import time
import itertools

import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray
import scipy.optimize
import tqdm
import xarray as xr

from fluxoniumcr import DATA_DIR

In [ ]:
# Construct Hamiltonian of two coupled fluxoniums
from fluxoniumcr.qubits.fluxonium import Fluxonium
from fluxoniumcr.qubits.product_basis import ProductBasis, DressedProductBasis


# Control qubit
Q0 = Fluxonium(
    EJ=5.60 * 2*pi,
    EC=1.87 * 2*pi,
    EL=0.56 * 2*pi,
    flux=0.5,
    dim=16,
    cutoff=128,
)

# Target qubit
Q1 = Fluxonium(
    EJ=3.52 * 2*pi,
    EC=1.18 * 2*pi,
    EL=0.88 * 2*pi,
    flux=0.5,
    dim=16,
    cutoff=128,
)

# Charge-charge coupling strength
JC = 97e-3 * 2*pi

prod_basis = ProductBasis([Q0, Q1])
prod_hamiltonian = (
    + sum(prod_basis.get_operator({i: 'hamiltonian'}) for i in range(2))
    + JC*prod_basis.get_operator(['charge', 'charge'])
)
dressed_basis = DressedProductBasis(
    hamiltonian=prod_hamiltonian,
    bare_basis=prod_basis,
    truncated_dims=(8, 4),
)

def E(*indices):
    index = dressed_basis.flat_index(indices)
    return dressed_basis.eigenvalues[index]

Ω0 = abs((Q0.eigenvalues[1] - Q0.eigenvalues[0])/Q0.get_operator('charge')[0, 1])
Q1_avg_freq = 0.5 * (E(0,1) - E(0,0) + E(1,1) - E(1,0))

In [ ]:
EJ0, EC0, EL0 = Q0.get_parameters(['EJ', 'EC', 'EL'])
EJ1, EC1, EL1 = Q1.get_parameters(['EJ', 'EC', 'EL'])

parent_path = (
    DATA_DIR
    /"time_domain"
    /(
        f"EJc={EJ0/(2*pi):.2f},ECc={EC0/(2*pi):.2f},ELc={EL0/(2*pi):.2f}"
        f",EJt={EJ1/(2*pi):.2f},ECt={EC1/(2*pi):.2f},ELt={EL1/(2*pi):.2f}"
        f",JC={JC/(2*pi):.3f}"
    )
)

In [ ]:
from injector import Injector

from fluxoniumcr.simulation.cnot_solver import (
    CNOTParameters,
    SmoothSquareParameters,
)
from fluxoniumcr.simulation.cnot_solver import CNOTDurationSweep, CNOTSolver
from fluxoniumcr.simulation.cnot_fidelity import (
    calculate_cnot_fidelity,
    calculate_cnot_offset,
    create_cross_resonance_unitary,
)
from fluxoniumcr.simulation.module import SimulationModule
from fluxoniumcr.simulation.operator_resolver import OperatorResolver
from fluxoniumcr.simulation.signals import planck_taper_signal, cosine_taper_signal


root = Injector([
    SimulationModule(
        basis=dressed_basis,
        names=['Q0', 'Q1'],
        dt=0.01,
    ),
])

cnot_solver = root.get(CNOTSolver)
op_resolver = root.get(OperatorResolver)

In [ ]:
def optimize_carrier_freq(
        p: CNOTParameters,
        *,
        cnot_solver: CNOTSolver,
) -> CNOTParameters:
    p = p.copy()
    
    def fobj(carrier_freq: float) -> float:
        p.pulse_parameters['carrier_freq'] = carrier_freq
        M = cnot_solver.solve(p)
        return 1 - calculate_cnot_fidelity(M)
    
    total_duration = p.pulse_parameters['total_duration']
    carrier_freq = p.pulse_parameters['carrier_freq']
    M = cnot_solver.solve(p)
    detuning_guess = -np.angle(M[1,1]/M[0,0] + M[3,3]/M[2,2])/(2*pi*total_duration)

    result = scipy.optimize.minimize_scalar(
        fobj,
        bracket=(carrier_freq, carrier_freq + detuning_guess),
        method='brent',
    )
    
    p.pulse_parameters['carrier_freq'] = result.x.item()
    
    return p


def optimize_total_duration_and_carrier_freq(
        p: CNOTParameters,
        *,
        cnot_solver: CNOTSolver,
) -> CNOTParameters:
    p = p.copy()
    
    def fobj(x: NDArray[np.floating]) -> float:
        total_duration, carrier_freq = x
        p.pulse_parameters['total_duration'] = total_duration
        p.pulse_parameters['carrier_freq'] = carrier_freq
        M = cnot_solver.solve(p)
        return 1 - calculate_cnot_fidelity(M)
    
    result = scipy.optimize.minimize(
        fobj,
        x0=(p.pulse_parameters['total_duration'],
            p.pulse_parameters['carrier_freq']),
        method='Nelder-Mead',
    )
    
    p.pulse_parameters['total_duration'] = result.x[0]
    p.pulse_parameters['carrier_freq'] = result.x[1]
    
    return p


def optimize_total_duration_shgo(
        p: CNOTParameters,
        *,
        bounds: tuple[float, float],
        n: int,
        cnot_solver: CNOTSolver,
) -> CNOTParameters:
    p = p.copy()
    
    sweep = cnot_solver.create_duration_sweep(p)
    
    result = scipy.optimize.shgo(
        functools.partial(optimize_total_duration_fobj, sweep=sweep),
        [p.pulse_parameters['total_duration'] + np.array(bounds)],
        n=n,
        sampling_method='sobol',
        minimizer_kwargs=dict(
            method='BFGS',
        ),
    )
    
    p.pulse_parameters['total_duration'] = result.x.item()
    
    return p


def optimize_total_duration_fobj(
        x: NDArray[np.floating],
        *,
        sweep: CNOTDurationSweep,
) -> float:
    total_duration = x[0]
    M = sweep.solve(total_duration)
    return 1 - calculate_cnot_fidelity(M)

In [ ]:
pulse_shape = 'planck_taper'

# ==================================

if pulse_shape == 'planck_taper':
    pulse_factory = planck_taper_signal
elif pulse_shape == 'cosine_taper':
    pulse_factory = cosine_taper_signal
else:
    raise ValueError(f"Unknown shape {pulse_shape!r}")


gate_parameters = CNOTParameters(
    pulse_parameters=SmoothSquareParameters(
        total_duration=0.0,
        ramp_duration=0.0,
        amplitude=0.0,
        carrier_freq=Q1_avg_freq,
    ),
    pulse_factory=pulse_factory,
    drive_operator="Q0.charge",
)

In [ ]:
# Sweep ramp duration for fixed amplitudes
amplitude_data = np.array([0.2, 0.4, 0.6, 0.8]) * Ω0
ramp_duration_data = np.concatenate([
    np.arange(1, 2, 0.0625),
    np.arange(2, 5, 0.125),
    np.arange(5, 10, 0.25),
    np.arange(10, 20, 0.5),
    np.arange(20, 50, 1.0),
    np.arange(50, 102, 2.0),
])
dataset_name = f"{pulse_shape}_ramp_duration_sweep"

# Sweep amplitude for a fixed ramp duration
amplitude_data = np.linspace(0.01, 1.2, 1191) * Ω0
ramp_duration_data = np.array([20])
dataset_name = f"{pulse_shape}_amplitude_sweep"

In [ ]:
dataset = xr.Dataset(
    attrs=dict(
        name=dataset_name,
        pulse_shape=pulse_shape,
    )
)

amplitude_coord = xr.DataArray(
    amplitude_data,
    dims=['amplitude'],
    attrs=dict(
        long_name="Midpoint pulse amplitude",
        units="Grad/s",
        plot_unit=Ω0,
    )
)
ramp_duration_coord = xr.DataArray(
    ramp_duration_data,
    dims=['ramp_duration'],
    attrs=dict(
        long_name="Ramp duration",
        units="ns",
    )
)
subsys_coord = xr.DataArray(
    ['control', 'target'],
    dims="subsystem",
    attrs=dict(long_name="Subsytem name")
)
state_coord = xr.DataArray(
    np.arange(32, dtype=np.uint8),
    dims="state",
    attrs=dict(long_name="Dressed state index")
)
initial_state_coord = xr.DataArray(
    np.arange(32, dtype=np.uint8),
    dims="initial_state",
    attrs=dict(long_name="Dressed state index (initial)")
)
final_state_coord = xr.DataArray(
    np.arange(32, dtype=np.uint8),
    dims="final_state",
    attrs=dict(long_name="Dressed state index (final)")
)

dataset['unitary'] = xr.DataArray(
    data=np.complex64(np.nan + 1j*np.nan),
    coords=[
        amplitude_coord,
        ramp_duration_coord,
        final_state_coord,
        initial_state_coord,
    ],
    attrs=dict(
        long_name="Optimized gate unitary",
    )
)

dataset['control_qubit_phase'] = xr.DataArray(
    np.float64('nan'),
    coords=[amplitude_coord, ramp_duration_coord],
    attrs=dict(
        long_name="Control qubit phase accumulation",
        units="degree",
    )
)

dataset['target_qubit_rotation'] = xr.DataArray(
    np.float64('nan'),
    coords=[amplitude_coord, ramp_duration_coord],
    attrs=dict(
        long_name="Cross-resonance IX angle",
        units="degree",
    )
)

dataset['fidelity'] = xr.DataArray(
    np.float64('nan'),
    coords=[amplitude_coord, ramp_duration_coord],
    attrs=dict(
        long_name="Average CNOT fidelity",
    )
)
dataset['total_duration'] = xr.DataArray(
    np.float64('nan'),
    coords=[amplitude_coord, ramp_duration_coord],
    attrs=dict(
        long_name="Total pulse duration",
        units="ns",
    )
)
dataset['frequency'] = xr.DataArray(
    np.float64('nan'),
    coords=[amplitude_coord, ramp_duration_coord],
    attrs=dict(
        long_name="Pulse carrier frequency",
        units="Grad/s",
    )
)

dataset['state_label'] = xr.DataArray(
    np.array(
        [dressed_basis.multi_index(i) for i in range(dressed_basis.dim)],
        np.uint8,
    ),
    coords=[state_coord, subsys_coord],
    attrs=dict(
        long_name="Maximum overlap bare state label",
    )
)

dataset['drive_operator'] = xr.DataArray(
    op_resolver.resolve(gate_parameters.drive_operator),
    coords=[final_state_coord, initial_state_coord],
    attrs=dict(
        long_name="Control fluxonium charge operator",
    )
)

dataset['hamiltonian'] = xr.DataArray(
    np.diag(op_resolver.H0),
    coords=[state_coord],
    attrs=dict(
        long_name="Eigenvalues of the time-independent Hamiltonian",
        units="Grad/s"
    )
)

initial_control_qubit_coord = xr.DataArray(
    np.arange(2, dtype=np.uint8),
    dims="initial_control_qubit",
    attrs=dict(long_name="Control qubit state (initial)")
)

final_control_qubit_coord = xr.DataArray(
    np.arange(2, dtype=np.uint8),
    dims="final_control_qubit",
    attrs=dict(long_name="Control qubit state (final)")
)

initial_target_qubit_coord = xr.DataArray(
    np.arange(2, dtype=np.uint8),
    dims="initial_target_qubit",
    attrs=dict(long_name="Target qubit state (initial)")
)

final_target_qubit_coord = xr.DataArray(
    np.arange(2, dtype=np.uint8),
    dims="final_target_qubit",
    attrs=dict(long_name="Target qubit state (final)")
)

dataset['matrix'] = xr.DataArray(
    data=np.complex64(np.nan + 1j*np.nan),
    coords=[
        amplitude_coord,
        ramp_duration_coord,
        final_control_qubit_coord,
        final_target_qubit_coord,
        initial_control_qubit_coord,
        initial_target_qubit_coord,
    ],
    attrs=dict(
        long_name="Unitary truncated to the computational subspace in the rotating frame",
    )
)

In [ ]:
def find_optimal_total_duration_and_carrier_freq(
        p: CNOTParameters,
        cnot_solver: CNOTSolver,
        last_total_duration: float|None = None,
) -> CNOTParameters:
    p = p.copy()
    
    total_duration_est = cnot_solver.estimate_cnot_total_duration(p)
    if abs(total_duration_est) > 1000:
        # Estimate failed, use last total_duration as an estimate instead
        if last_total_duration is None:
            print(f"{total_duration_est=} and {last_total_duration=}!")
            last_total_duration = 200  # Take a wild guess
        total_duration_est = last_total_duration
    
    p.pulse_parameters['total_duration'] = total_duration_est

    p = optimize_carrier_freq(p, cnot_solver=cnot_solver)
    
    ramp_duration = p.pulse_parameters['ramp_duration']
    if ramp_duration >= 20:
        p = optimize_total_duration_and_carrier_freq(p, cnot_solver=cnot_solver)
    else:
        if ramp_duration >= 10:
            bounds = (-5.0, 5.0)
        elif ramp_duration >= 5:
            bounds = (-10.0, 10.0)
        else:
            bounds = (-40.0, 40.0)
        
        for _ in range(2):
            p = optimize_total_duration_shgo(
                p,
                bounds=bounds,
                n=round(2*(bounds[1] - bounds[0])),
                cnot_solver=cnot_solver,
            )
            p = optimize_carrier_freq(p, cnot_solver=cnot_solver)
            
        p = optimize_total_duration_and_carrier_freq(p, cnot_solver=cnot_solver)
        
    return p

In [ ]:
last_total_duration = None


for amplitude in dataset.amplitude.data:
    print(f"Amplitude: {amplitude/Ω0:.4g} Ω0")
    for ramp_duration in tqdm.tqdm(dataset.ramp_duration.data):
        ds = dataset.loc[dict(amplitude=amplitude, ramp_duration=ramp_duration)]
        
        if not math.isnan(ds.fidelity.item()):
            last_total_duration = ds.total_duration.item()
            continue
            
        p = gate_parameters.copy()
        p.pulse_parameters['amplitude'] = amplitude
        p.pulse_parameters['ramp_duration'] = ramp_duration
        
        p = find_optimal_total_duration_and_carrier_freq(
            p,
            cnot_solver=cnot_solver,
            last_total_duration=last_total_duration,
        )
        
        ds.total_duration[()] = p.pulse_parameters['total_duration']
        ds.frequency[()] = p.pulse_parameters['carrier_freq']

        U = cnot_solver.solve(p, return_unitary=True)
        ds.unitary[...] = U

        M = cnot_solver.solve(p)
        theta, phi = calculate_cnot_offset(M)
        cnot_fidelity = calculate_cnot_fidelity(M)

        ds.control_qubit_phase[()] = np.rad2deg(theta)
        ds.target_qubit_rotation[()] = np.rad2deg(phi)
        ds.fidelity[()] = cnot_fidelity
        ds.matrix[...] = M.reshape(2, 2, 2, 2)
        
        last_total_duration = p.pulse_parameters['total_duration']

In [ ]:
# Save dataset
dataset.to_netcdf(parent_path/(dataset.attrs['name'] + ".hdf5"), engine="h5netcdf")